<a href="https://colab.research.google.com/github/ryusko-alt/MLB-Analytical-Projects-RY/blob/main/MLB_Postseason_Strength_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MLB Postseason Strength Model
### Underlying-performance-based Postseason Strength Score (0-100), series win probabilities, and championship simulation

This notebook scores playoff teams on **process metrics** (the stats that predict future performance) rather than
runs/wins already banked, on the premise that postseason success is better predicted by *true talent* indicators
than by regular-season record.

**Categories & metrics:**
- **Offense:** wRC+, BB%+ minus K%+ differential
- **Pitching:** xFIP-, K-BB%, Stuff+, Pitching+, Location+
- **Defense:** DRS
- **Baserunning:** BsR

All weights are adjustable in the Configuration cell below (same pattern as your other sensitivity-tab workbooks).

In [1]:
# ============================================================
# CONFIGURATION - adjust weights here, everything downstream updates
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
from math import comb

pd.set_option('display.float_format', lambda x: f'{x:.2f}')

# ---- Category weights (must sum to 1.0) ----
CATEGORY_WEIGHTS = {
    'offense':     0.35,
    'pitching':    0.40,   # pitching weighted heaviest: short-series outcomes lean on run prevention
    'defense':     0.15,
    'baserunning': 0.10,
}

# ---- Sub-metric weights within each category (each group sums to 1.0) ----
OFFENSE_WEIGHTS = {
    'wRC+':           0.70,
    'BB_K_plus_diff': 0.30,
}

PITCHING_WEIGHTS = {
    'xFIP_minus':    0.25,
    'K_BB_pct':      0.25,
    'Stuff_plus':    0.20,
    'Pitching_plus': 0.15,
    'Location_plus': 0.15,
}

# Defense (DRS) and Baserunning (BsR) are single-metric categories

# ---- Direction flags: True = higher raw value is better ----
METRIC_DIRECTION = {
    'wRC+': True,
    'BB_K_plus_diff': True,
    'xFIP_minus': False,     # lower xFIP- is better
    'K_BB_pct': True,
    'Stuff_plus': True,
    'Pitching_plus': True,
    'Location_plus': True,
    'DRS': True,
    'BsR': True,
}

# Logistic scale for converting score gaps into single-game win probability.
# Larger ELO_SCALE = a given score gap produces a *smaller* win-prob edge.
ELO_SCALE = 12.0

assert abs(sum(CATEGORY_WEIGHTS.values()) - 1.0) < 1e-6, "Category weights must sum to 1.0"
assert abs(sum(OFFENSE_WEIGHTS.values()) - 1.0) < 1e-6, "Offense weights must sum to 1.0"
assert abs(sum(PITCHING_WEIGHTS.values()) - 1.0) < 1e-6, "Pitching weights must sum to 1.0"
print("Config OK.")

Config OK.


## Data Input

Fill this in with the confirmed playoff field. Recommended source: FanGraphs team pages.
- wRC+, BB%+, K%+ → Team Stats → Batting → Advanced
- xFIP-, K-BB%, Stuff+/Location+/Pitching+ → Team Stats → Pitching / Pitch Info
- DRS → Team Stats → Fielding
- BsR → Team Stats → BaseRunning

Replace the placeholder rows below with the real teams and values once the field is set.

In [3]:
# ============================================================
# DATA INPUT - replace with real playoff-team metrics
# ============================================================
data = {
    'Team':          ['Dodgers', 'Blue Jays', 'Brewers', 'Mariners'],
    'wRC+':          [113, 112, 107, 113],
    'BB_pct_plus':   [110, 102, 106, 106],   # BB%+
    'K_pct_plus':    [100, 79, 93, 103],   # K%+ (lower is better for hitters, handled via the diff)
    'xFIP_minus':    [96, 99, 99, 94],
    'K_BB_pct':      [0.18, 0.14, 0.16, 0.10],
    'Stuff_plus':    [105, 98, 110, 95],
    'Pitching_plus': [103, 100, 108, 97],
    'Location_plus': [101, 99, 104, 96],
    'DRS':           [42, 15, 38, -5],
    'BsR':           [8.5, 2.0, 5.5, -3.0],
}

df = pd.DataFrame(data).set_index('Team')
df['BB_K_plus_diff'] = df['BB_pct_plus'] - df['K_pct_plus']
df

,wRC+,BB_pct_plus,K_pct_plus,xFIP_minus,K_BB_pct,Stuff_plus,Pitching_plus,Location_plus,DRS,BsR,BB_K_plus_diff
Team,,,,,,,,,,,
Team A,115,105,92,88,0.18,105,103,101,42,8.50,13
Team B,108,98,101,95,0.14,98,100,99,15,2.00,-3
Team C,122,112,88,91,0.16,110,108,104,38,5.50,24
Team D,98,95,105,102,0.10,95,97,96,-5,-3.00,-10


## Normalization & Scoring Engine

Each metric is min-max scaled to 0-100 **across the current playoff-team pool** (relative strength, not
absolute), with lower-is-better metrics (xFIP-) auto-inverted via `METRIC_DIRECTION`.

In [4]:
def normalize_column(series, higher_is_better=True):
    """Min-max scale a metric to 0-100 across the playoff-team pool."""
    lo, hi = series.min(), series.max()
    if hi == lo:
        return pd.Series(100.0, index=series.index)
    scaled = (series - lo) / (hi - lo) * 100
    return scaled if higher_is_better else (100 - scaled)


def compute_category_score(frame, metric_weights):
    """Weighted average of normalized metrics for one category."""
    norm_cols = {m: normalize_column(frame[m], METRIC_DIRECTION[m]) for m in metric_weights}
    norm_df = pd.DataFrame(norm_cols)
    weights = pd.Series(metric_weights)
    return norm_df.mul(weights, axis=1).sum(axis=1)


offense_score     = compute_category_score(df, OFFENSE_WEIGHTS)
pitching_score    = compute_category_score(df, PITCHING_WEIGHTS)
defense_score     = normalize_column(df['DRS'], True)
baserunning_score = normalize_column(df['BsR'], True)

scores = pd.DataFrame({
    'Offense':     offense_score,
    'Pitching':    pitching_score,
    'Defense':     defense_score,
    'Baserunning': baserunning_score,
})

scores['Postseason Strength Score'] = (
    scores['Offense']     * CATEGORY_WEIGHTS['offense']     +
    scores['Pitching']    * CATEGORY_WEIGHTS['pitching']    +
    scores['Defense']     * CATEGORY_WEIGHTS['defense']     +
    scores['Baserunning'] * CATEGORY_WEIGHTS['baserunning']
)

ranked = scores.sort_values('Postseason Strength Score', ascending=False)
ranked

,Offense,Pitching,Defense,Baserunning,Postseason Strength Score
Team,,,,,
Team C,100.00,88.39,91.49,73.91,91.47
Team A,69.88,80.89,100.00,100.00,81.81
Team B,35.34,38.72,42.55,43.48,38.59
Team D,0.00,0.00,0.00,0.00,0.00


In [ ]:
plt.figure(figsize=(9, 5))
plt.barh(ranked.index, ranked['Postseason Strength Score'], color='#2b6cb0')
plt.xlabel('Postseason Strength Score (0-100)')
plt.title('MLB Postseason Strength Rankings')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Series Win Probability

Single-game win probability is derived logistically from the score gap between two teams (Elo-style).
Series win probability then comes from the standard negative-binomial series formula for a best-of-N
series, rather than just multiplying game odds — this correctly accounts for a team only needing to win
the majority of games, not all of them.

In [5]:
def series_game_win_prob(score_a, score_b, elo_scale=ELO_SCALE):
    """Single-game win probability for Team A vs Team B."""
    diff = score_a - score_b
    return 1 / (1 + 10 ** (-diff / elo_scale))


def series_win_prob(score_a, score_b, best_of=7, elo_scale=ELO_SCALE):
    """Probability Team A wins a best-of-N series (independent games at a fixed win prob)."""
    p = series_game_win_prob(score_a, score_b, elo_scale)
    wins_needed = best_of // 2 + 1
    prob_a_wins = 0.0
    for losses in range(wins_needed):
        games = wins_needed + losses
        prob_a_wins += comb(games - 1, losses) * (p ** wins_needed) * ((1 - p) ** losses)
    return prob_a_wins


# Example: quick head-to-head matrix for every pair in the field, best-of-7
teams = ranked.index.tolist()
matchup_matrix = pd.DataFrame(index=teams, columns=teams, dtype=float)
for a in teams:
    for b in teams:
        if a == b:
            matchup_matrix.loc[a, b] = np.nan
        else:
            matchup_matrix.loc[a, b] = round(
                series_win_prob(scores.loc[a, 'Postseason Strength Score'],
                                 scores.loc[b, 'Postseason Strength Score'], best_of=7) * 100, 1)

matchup_matrix

,Team C,Team A,Team B,Team D
Team C,NaN,99.20,100.00,100.00
Team A,0.80,NaN,100.00,100.00
Team B,0.00,0.00,NaN,100.00
Team D,0.00,0.00,0.00,NaN


## Championship Probability - Monte Carlo Bracket Simulation

Define the bracket as nested round-by-round pairings. Use `None` as a placeholder opponent for a team
that has a bye in that round (e.g., top division-winner seeds skipping the Wild Card round).

`best_of_map` sets series length per round index (0 = Wild Card, 1 = Division Series, 2 = Championship
Series, 3 = World Series) — adjust to match the confirmed postseason structure.

In [6]:
def simulate_series(team_a, team_b, scores_dict, best_of=7, elo_scale=ELO_SCALE):
    if team_b is None:
        return team_a
    if team_a is None:
        return team_b
    p = series_win_prob(scores_dict[team_a], scores_dict[team_b], best_of, elo_scale)
    return team_a if random.random() < p else team_b


def simulate_bracket(bracket, scores_dict, n_sims=20000, best_of_map=None):
    """
    bracket: list of first-round (team_a, team_b_or_None) pairs, e.g.
        [('Team A', None), ('Team B', 'Team C'), ('Team D', 'Team E'), ('Team F', None)]
    Returns championship percentage per team across n_sims simulated brackets.
    """
    if best_of_map is None:
        best_of_map = {0: 3, 1: 5, 2: 7, 3: 7}

    champ_counts = {t: 0 for pair in bracket for t in pair if t is not None}

    for _ in range(n_sims):
        current_round = [list(pair) for pair in bracket]
        round_idx = 0
        while len(current_round) > 1 or len(current_round[0]) > 1:
            winners = []
            for pair in current_round:
                bo = best_of_map.get(round_idx, 7)
                a = pair[0] if len(pair) > 0 else None
                b = pair[1] if len(pair) > 1 else None
                winners.append(simulate_series(a, b, scores_dict, bo))
            if len(winners) == 1:
                champ_counts[winners[0]] += 1
                break
            current_round = [winners[i:i + 2] for i in range(0, len(winners), 2)]
            round_idx += 1

    return (pd.Series(champ_counts) / n_sims * 100).sort_values(ascending=False)


scores_dict = scores['Postseason Strength Score'].to_dict()

# Example bracket - replace with the real confirmed field/seeding
example_bracket = [('Team A', None), ('Team B', 'Team C')]

champ_probs = simulate_bracket(example_bracket, scores_dict, n_sims=20000,
                                best_of_map={0: 3, 1: 5, 2: 7})
champ_probs

,0
Team C,98.03
Team A,1.97
Team B,0.00


## Notes / Next Steps

- **Metric pool:** because normalization is min-max *within the current playoff field*, scores are
  relative strength among this year's teams, not an absolute talent scale — re-run once the full field
  is set for stable rankings.
- **ELO_SCALE:** currently a reasonable starting assumption. If you want it calibrated (e.g. backtested
  against 5-10 years of actual postseason series results so implied win probabilities match historical
  base rates), that's a natural next iteration.
- **Bracket structure:** the simulator supports byes but assumes a simple fixed bracket. Real MLB
  postseason re-seeding (remaining higher seed always plays remaining lowest seed) would need a small
  addition to `simulate_bracket` if you want that exact behavior modeled — happy to add it once the
  actual field and seeding rules for this postseason are set.